# O1 - Data preprocessing (Grant Witness)
data source: grant-witness.us

### Imports

In [27]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

In [28]:
import pandas as pd
import re

from general.util.banned_words import add_flagged_column_vectorized

Reading data

In [29]:
dfs = ['cdc', 'epa', 'nih', 'nsf', 'samhsa']

In [30]:
for name in dfs:
    globals()[f'df_{name}'] = pd.read_csv(f'00_data/{name}_terminations.csv',encoding='latin1')
    print(name, globals()[f'df_{name}'].shape)

cdc (542, 30)
epa (671, 30)
nih (5836, 56)
nsf (1996, 40)
samhsa (2931, 32)


## Finding uniform columns

In [5]:
cols = {}

for name in dfs:
    df = globals()[f'df_{name}']
    cols[name] = sorted(df.columns)

shared_all_before = set(cols[dfs[0]])
for name in dfs[1:]:
    shared_all_before &= set(cols[name])

print(shared_all_before)


{'usaspending_url', 'org_city', 'status', 'termination_date', 'org_state', 'reinstatement_indicator'}


Renaming

In [6]:
all_columns = set()

for name in dfs:
    df = globals()[f"df_{name}"]
    all_columns |= set(df.columns)

all_columns = sorted(all_columns)

In [7]:
overview = pd.DataFrame(index=all_columns, columns=dfs)

for name in dfs:
    df = globals()[f"df_{name}"]
    overview[name] = overview.index.isin(df.columns).astype(int)

In [8]:
overview

,cdc,epa,nih,nsf,samhsa
abstract,0,0,0,1,1
abstract_text,0,0,1,0,0
activity_code,0,0,1,0,0
activity_type,1,0,0,0,1
appl_id,0,0,1,0,0
...,...,...,...,...,...
usasp_outlaid,0,0,0,1,0
usasp_start_date,0,0,0,1,0
usasp_total_obligated,0,0,0,1,0
usasp_total_obligated_corrected,0,0,0,1,0


In [ ]:
#overview.to_excel('columns.xlsx')

In [9]:
# EPA
df_epa = df_epa.rename(columns={
    'organization': 'org_name',
    'original_end_date': 'end_date_original',
    'project_description': 'abstract',
    'award_value': 'award_amount',
    'start_date': 'start_date_original'
})

# NSF
df_nsf = df_nsf.rename(columns={
    'nsf_end_date': 'end_date_original',
    'nsf_start_date': 'start_date_original',
    'nsf_total_budget': 'award_amount',
    'estimated_remaining' : 'award_remaining', 
    'estimated_outlays': 'award_outlaid'
})

# NIH
df_nih = df_nih.rename(columns={
    'core_award_number': 'grant_id',
    'reinstated_est_date': 'reinstatement_date',
    'project_end_date': 'end_date_original',
    'abstract_text': 'abstract',
    'total_award': 'award_amount',
    'targeted_start_date': 'start_date_original',
    'total_estimated_remaining' : 'award_remaining',
    'total_estimated_outlays': 'award_outlaid'
})

# CDC
df_cdc = df_cdc.rename(columns={
    'title': 'project_title',
    'award_value': 'award_amount',
    'project_start_date': 'start_date_original'
})

# SAMHSA
df_samhsa = df_samhsa.rename(columns={
    'title': 'project_title',
    'award_value': 'award_amount',
    'project_start_date': 'start_date_original'
})

In [10]:
# to make sure i know which is which
df_epa['agency'] = 'EPA'
df_nsf['agency'] = 'NSF'
df_nih['agency'] = 'NIH'
df_cdc['agency'] = 'CDC'
df_samhsa['agency'] = 'SAMHSA'

Verification

In [11]:
cols = {}

for name in dfs:
    df = globals()[f'df_{name}']
    cols[name] = sorted(df.columns)

shared_all_after = set(cols[dfs[0]])
for name in dfs[1:]:
    shared_all_after &= set(cols[name])

print(sorted(shared_all_after))


['agency', 'award_amount', 'award_outlaid', 'award_remaining', 'grant_id', 'org_city', 'org_name', 'org_state', 'project_title', 'reinstatement_date', 'reinstatement_indicator', 'start_date_original', 'status', 'termination_date', 'usaspending_url']


## Cleaning status

In [12]:
dfs = {'nih': df_nih, 'nsf': df_nsf, 'epa': df_epa, 'samhsa': df_samhsa, 'cdc': df_cdc}

In [13]:
def fix_status(s: str) -> str:
    if s is None:
        return s
    s = str(s)

    # 1) Try to repair mojibake (UTF-8 read as latin1/cp1252)
    try:
        s = s.encode("latin1").decode("utf-8")
    except Exception:
        pass

    # 2) Remove leading emoji / bullets / weird prefixes, keep the readable status text
    #    Keep letters, numbers, spaces, and hyphens.
    s = re.sub(r"[^A-Za-z0-9\s\-]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    return s

for name in dfs:
    if "status" in dfs[name].columns:
        dfs[name]["status"] = dfs[name]["status"].apply(fix_status)

In [14]:
for name in dfs:
    if "status" in dfs[name].columns:
        print(name)
        print(dfs[name]["status"].dropna().head(10).tolist())

nih
['Unfrozen Funding', 'Unfrozen Funding', 'Possibly Unfrozen Funding', 'Possibly Reinstated', 'Possibly Reinstated', 'Terminated', 'Terminated', 'Possibly Unfrozen Funding', 'Possibly Unfrozen Funding', 'Possibly Reinstated']
nsf
['Terminated', 'Terminated', 'Terminated', 'Terminated', 'Terminated', 'Terminated', 'Terminated', 'Terminated', 'Terminated', 'Terminated']
epa
['Terminated', 'Terminated', 'Terminated', 'Terminated', 'Terminated', 'Terminated', 'Terminated', 'Terminated', 'Terminated', 'Terminated']
samhsa
['Possibly Reinstated', 'Possibly Reinstated', 'Possibly Reinstated', 'Possibly Reinstated', 'Possibly Reinstated', 'Possibly Reinstated', 'Possibly Reinstated', 'Possibly Reinstated', 'Possibly Reinstated', 'Possibly Reinstated']
cdc
['Possibly Reinstated', 'At-risk', 'Possibly Reinstated', 'Possibly Reinstated', 'Possibly Reinstated', 'Possibly Reinstated', 'Possibly Reinstated', 'Possibly Reinstated', 'Possibly Reinstated', 'Possibly Reinstated']


In [15]:
df_samhsa

,grant_id,status,event_history,project_title,termination_date,termination_indicator,reinstatement_date,reinstatement_indicator,org_name,abstract,...,assistance_listing_title,funding_office_name,cfda_number,cfda_title,nofo,nofo_title,usaspending_url,taggs_url,nofo_url,agency
0,FG001136,Possibly Reinstated,- 2023-08-28: Grant awarded\n- 2023-08-28: Obl...,FY 2023 Cooperative Agreement for the Hispanic...,2026-01-13,HHS TAGGS,2026-01-14,Other reporting,UNIVERSIDAD CENTRAL DEL CARIBE,The Hispanic/Latino Behavioral Health CoE (H/L...,...,Substance Abuse and Mental Health Services Pro...,SAMHSA CENTER FOR SUBSTANCE ABUSE TREATMENT,93.243,SUBSTANCE ABUSE AND MENTAL HEALTH SERVICES PRO...,FG-23-002,Cooperative Agreement for the Hispanic/Latino ...,https://www.usaspending.gov/award/ASST_NON_H79...,https://taggs.hhs.gov/Detail/AwardDetail?arg_A...,https://www.samhsa.gov/grants/grant-announceme...,SAMHSA
1,FG001142,Possibly Reinstated,- 2023-08-28: Grant awarded\n- 2023-08-28: Obl...,FY 2023 American Indian and Alaska Native Beha...,2026-01-14,HHS TAGGS,2026-01-15,Other reporting,UNIVERSITY OF ARIZONA,The overall purpose of the University of Arizo...,...,Substance Abuse and Mental Health Services Pro...,SAMHSA CENTER FOR MENTAL HEALTH SERVICES,93.243,SUBSTANCE ABUSE AND MENTAL HEALTH SERVICES PRO...,FG-23-001,American Indian and Alaska Native Behavioral H...,https://www.usaspending.gov/award/ASST_NON_H79...,https://taggs.hhs.gov/Detail/AwardDetail?arg_A...,https://www.samhsa.gov/grants/grant-announceme...,SAMHSA
2,FG001251,Possibly Reinstated,- 2023-09-25: Grant awarded\n- 2023-09-25: Obl...,"Family Counseling and Support for Lesbian, Gay...",2026-01-13,HHS TAGGS,2026-01-14,Other reporting,"HOME FOR LITTLE WANDERERS, INC., THE",Out of Home Youth &Family Support Project offe...,...,Substance Abuse and Mental Health Services Pro...,SAMHSA OFFICE OF THE ASSITANT SECRETARY FOR ME...,93.243,SUBSTANCE ABUSE AND MENTAL HEALTH SERVICES PRO...,FG-23-004,Family Support,https://www.usaspending.gov/award/ASST_NON_H79...,https://taggs.hhs.gov/Detail/AwardDetail?arg_A...,https://www.samhsa.gov/grants/grant-announceme...,SAMHSA
3,FG001252,Possibly Reinstated,- 2023-09-25: Grant awarded\n- 2023-09-25: Obl...,"Family Counseling and Support for Lesbian, Gay...",2026-01-13,HHS TAGGS,2026-01-14,Other reporting,"CENTERSTONE OF TENNESSEE, INC.",Centerstoneâs LGBTQI+ Family Support Program...,...,Substance Abuse and Mental Health Services Pro...,SAMHSA CENTER FOR MENTAL HEALTH SERVICES,93.243,SUBSTANCE ABUSE AND MENTAL HEALTH SERVICES PRO...,FG-23-004,Family Support,https://www.usaspending.gov/award/ASST_NON_H79...,https://taggs.hhs.gov/Detail/AwardDetail?arg_A...,https://www.samhsa.gov/grants/grant-announceme...,SAMHSA
4,FG001272,Possibly Reinstated,- 2023-09-25: Grant awarded\n- 2023-09-25: Obl...,"Family Counseling and Support for Lesbian, Gay...",2026-01-13,HHS TAGGS,2026-01-14,Other reporting,UNIVERSITY OF ARIZONA,Family Pride Initiative is a collaboration bet...,...,Substance Abuse and Mental Health Services Pro...,SAMHSA CENTER FOR MENTAL HEALTH SERVICES,93.243,SUBSTANCE ABUSE AND MENTAL HEALTH SERVICES PRO...,FG-23-004,Family Support,https://www.usaspending.gov/award/ASST_NON_H79...,https://taggs.hhs.gov/Detail/AwardDetail?arg_A...,https://www.samhsa.gov/grants/grant-announceme...,SAMHSA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2926,TI088245,Possibly Reinstated,- 2025-09-24: Grant awarded or renewed\n- 2025...,Targeted Capacity Expansion: Special Projects,2026-01-13,HHS TAGGS,2026-01-14,Other reporting,"PROGRAMA GUARA BI D/B/A GUARA BI, INC.",NaN,...,Substance Abuse and Mental Health Services Pro...,SAMHSA CENTER FOR SUBSTANCE ABUSE TREATMENT,93.243,SUBSTANCE ABUSE AND MENTAL HEALTH SERVICES PRO...,TI-25-002,Targeted Capacity Expansion: Special Projects,https://www.usaspending.gov/award/ASST_NON_H79...,https://taggs.hhs.gov/Detail/AwardDetail?arg_A...,https://www.samhsa.gov/grants/grant-announceme...,SAM

## Missing values overview

In [16]:
na_share = df_samhsa.isna().mean()

for col, val in na_share[na_share > 0].sort_values(ascending=False).items():
    print(f"{col}: {val:.4f}")

abstract: 0.0894
reinstatement_date: 0.0457
reinstatement_indicator: 0.0457
award_outlaid: 0.0437
award_remaining: 0.0437
start_date_original: 0.0007
project_original_end_date: 0.0007
award_amount: 0.0007
funding_office_name: 0.0007
cfda_number: 0.0007
cfda_title: 0.0007
usaspending_url: 0.0007
nofo_title: 0.0003
nofo_url: 0.0003


Getting rid of CDC as it has no abstract

In [18]:
dfs.remove('cdc')

AttributeError: 'dict' object has no attribute 'remove'

## Creating column with flagged words

In [ ]:
df_epa = add_flagged_column_vectorized(df_epa)
df_nih = add_flagged_column_vectorized(df_nih)
df_nsf = add_flagged_column_vectorized(df_nsf)
df_samhsa = add_flagged_column_vectorized(df_samhsa)
df_cdc = add_flagged_column_vectorized(df_cdc)

Checking if worked

In [26]:
df_nih.columns

Index(['status', 'grant_id', 'full_award_number', 'hhs_web_reported',
       'hhs_pdf_reported', 'self_reported', 'court_reported',
       'source_reported', 'start_date_original', 'targeted_end_date',
       'ever_frozen', 'frozen_date', 'unfrozen_date', 'file_c_outlays',
       'termination_date', 'cancellation_source', 'reinstatement_indicator',
       'reinstatement_date', 'reinstatement_case', 'last_payment_month',
       'last_payment_date', 'project_title', 'activity_code', 'org_name',
       'org_type', 'dept_type', 'program_office', 'org_state', 'org_city',
       'org_congdist', 'us_rep', 'us_rep_phone', 'flagged_words',
       'study_section', 'foa', 'foa_title', 'abstract', 'phr_text', 'terms',
       'award_amount', 'award_outlaid', 'award_remaining',
       'spending_categories', 'notes', 'org_traits', 'pct_ugrad_pellgrant',
       'pct_ugrad_fedloan', 'appl_id', 'prog_office_code', 'funding_category',
       'nih_activity', 'court_restoration_url', 'usaspending_url', 'ta

## Saving

In [25]:
df_epa.to_csv('00_data/01_epa.csv')
df_nih.to_csv('00_data/01_nih.csv')
df_nsf.to_csv('00_data/01_nsf.csv')
df_samhsa.to_csv('00_data/01_samhsa.csv')
df_cdc.to_csv('00_data/01_cdc.csv')


# Next: 02_embeddings